In [82]:
#Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as sig
from scipy.stats import ttest_ind
import mne

from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline



In [83]:

#This function takes in a gdf file path and converts it to a pandas dataframe
def path_to_dataframe(FILE_PATH, fs):
    eog = ['EOG-left', 'EOG-central', 'EOG-right']

    #read raw data from gdf using mne
    file_raw = mne.io.read_raw_gdf(FILE_PATH, eog=eog)
    eval_raw = mne.io.read_raw_gdf(EVAL_PATH, eog=eog)

    #read events from the gdf annotation
    events, event_id = mne.events_from_annotations(file_raw)
    

    #convert raw to dataframe
    eeg_data = file_raw.to_data_frame()
    eeg_data = eeg_data.drop(eeg_data.columns[0], axis=1)
    eeg_data = eeg_data.drop(eeg_data.columns[-1], axis=1)
    eeg_data = eeg_data.drop(eeg_data.columns[-1], axis=1)
    eeg_data = eeg_data.drop(eeg_data.columns[-1], axis=1)
    eeg_data['event'] = 0


    #convert events to dataframe
    events_df = pd.DataFrame(events)
    events_df = events_df.drop(events_df.columns[1], axis=1)

    #iterates through events and creates a time-matched list to correspond events with eeg timestamps
    track = 0
    hold_size = len(eeg_data)
    hold_list = [0] * hold_size
    for i in events_df[0]:
        time_check = i
        hold_list[i] = events_df.at[track, 2]
        track = track + 1

    #iterates through hold_list (which has markers for the start of trials/ events) and populates a list with labels to associate with eeg timestamps
    one_count = 0
    two_count = 0
    three_count = 0
    four_count = 0
    five_count = 0
    six_count = 0
    seven_count = 0
    eight_count = 0
    nine_count = 0
    ten_count = 0
    hold_two = hold_list.copy()
    for i in range(len(hold_list)):
    #Correspondence of numbers dictionary
    #1023: 1, Rejected trial
    #1072: 2, Eye movements
    #276: 3, Idling EEG (eyes open)
    #277: 4, Idling EEG (eyes closed)
    #32766: 5, Start of a new run
    #768: 6, Start of a trial
    #769: 7, Cue onset left (class 1)
    #770: 8, Cue onset right (class 2)
    #771: 9, Cue onset foot (class 3)
    #772: 10, Cue onset tongue (class 4)
    
        if hold_list[i] == 1:
            #1 indicates a rejected trial (See above "dictionary")
            #Trials are roughly 6 seconds
            #for 250 Hz, this means a trial is approximately 1500 samples in duration

            #Upon seeing a '1', or rejected trial, set the next 6 seconds to '0'
            hold_two[i:(i+(fs*6))] = [0] * (fs*6)
            #Counters to numerate the appearance of each class, primarily for debugging purposes
            one_count = one_count + 1
        elif hold_list[i] == 2:
            #2-4 are irrelevant, also set to '0'
            hold_two[i] = 0
            two_count = two_count + 1
        elif hold_list[i] == 3:
            hold_two[i] = 0
            three_count = three_count + 1
        elif hold_list[i] == 4:
            hold_two[i] = 0
            four_count = four_count + 1
        elif hold_list[i] == 5:
            #5 indicates a new run, starting after a rest period
            #Also irrelevant to classification, so set to '0'
            hold_two[i] = 0
            five_count = five_count + 1
    
        elif hold_list[i] == 6:
            #6 indicates the start of the 6 second trial window
            #Motor Imagery should begin 2 seconds after every 6 second window begins (500 samples)
            #Since each MI task is accurately recorded, this also can be set to '0'
            hold_two[i] = 0
            six_count = six_count + 1
        elif hold_list[i] == 7:
            #7 Indicates left hand MI, or 'class 1'
            #Each motor Imagery task should last ~4 seconds, though the 'end' is not indicated within the data
            #Seeing a 7 will, similarly to rejected trial above, set the next 4 seconds to a value of '1'
            hold_two[i:(i+(fs*4))] = [1] * (fs*4)
            seven_count = seven_count + 1
        if hold_list[i] == 8:
            #8 indicates right hand MI, 'class 2'
            #Set next 4 seconds to '2'
            hold_two[i:(i+(fs*4))] = [2] * (fs*4)
            eight_count = eight_count + 1
        if hold_list[i] == 9:
            #9 - Foot MI
            #Set to '3'
            hold_two[i:(i+(fs*4))] = [3] * (fs*4)
            nine_count = nine_count + 1
        if hold_list[i] == 10:
            #10 - Tongue MI
            #Set to '4'
            hold_two[i:(i+(fs*4))] = [4] * (fs*4)
            ten_count = ten_count + 1
#print(one_count)
#print(two_count)
#print(three_count)
#print(four_count)
#print(five_count)
#print(six_count)
#print(seven_count)
#print(eight_count)
#print(nine_count)
#print(ten_count)
    eeg_data['event'] = hold_two
    return eeg_data


In [84]:
#This function takes in a dataframe, and converts it for use in the 'identifier' algorithm
#Assumes the nature of the dataframe from 'path_to_dataframe'
def dataframe_to_nparr(df):
    arr_temp = np.asarray(df).T
    arr_1 = np.delete(arr_1_temp, 22, axis=0)
    return arr_1

In [274]:
freq = 250
FILE_PATH_1 = 'C:/Users/semee/Class/EEEE 536/Project/Data/A01T.gdf'
FILE_PATH_2 = 'C:/Users/semee/Class/EEEE 536/Project/Data/A02T.gdf'
FILE_PATH_3 = 'C:/Users/semee/Class/EEEE 536/Project/Data/A03T.gdf'
FILE_PATH_4 = 'C:/Users/semee/Class/EEEE 536/Project/Data/A04T.gdf'
FILE_PATH_5 = 'C:/Users/semee/Class/EEEE 536/Project/Data/A05T.gdf'

subj1 = path_to_dataframe(FILE_PATH_1, freq)
subj2 = path_to_dataframe(FILE_PATH_2, freq)
#subj3 = path_to_dataframe(FILE_PATH_3, freq)
#subj4 = path_to_dataframe(FILE_PATH_4, freq)
#subj5 = path_to_dataframe(FILE_PATH_5, freq)

arr_1_x = dataframe_to_nparr(subj1)
arr_1_y = np.ones(22) * 0
arr_2_x = dataframe_to_nparr(subj2)
arr_2_y = np.ones(22)
arr_t_x = np.concatenate((arr_1_x, arr_2_x))
arr_t_y = np.concatenate((arr_1_y, arr_2_y))
arr_t_x = arr_t_x[:,100000:108192]

#X_train, X_test, y_train, y_test = train_test_split(
    #arr_t_x, arr_t_y, test_size=0.20, random_state=42)
print(type(arr_t_x))
X_tensor = torch.FloatTensor(arr_t_x[:, np.newaxis, np.newaxis, :])
y_tensor = torch.LongTensor(arr_t_y)

print(f'Input tensor shape: {X_tensor.shape}')
print(f'Labels: {y_tensor.unique(return_counts=True)}')

n_train = int(0.7 * len(X_tensor))
indices = torch.randperm(len(X_tensor))
train_idx = indices[:n_train]
test_idx = indices[n_train:]

train_dataset = TensorDataset(X_tensor[train_idx], y_tensor[train_idx])
test_dataset = TensorDataset(X_tensor[test_idx], y_tensor[test_idx])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

Extracting GDF parameters from C:/Users/semee/Class/EEEE 536/Project/Data/A01T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
Extracting GDF parameters from C:/Users/semee/Class/EEEE 536/Project/Data/A01E.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('771'), np.str_('772')]


C:\Users\semee\AppData\Local\Python\pythoncore-3.14-64\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\Users\semee\AppData\Local\Python\pythoncore-3.14-64\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Extracting GDF parameters from C:/Users/semee/Class/EEEE 536/Project/Data/A02T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
Extracting GDF parameters from C:/Users/semee/Class/EEEE 536/Project/Data/A01E.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('771'), np.str_('772')]


C:\Users\semee\AppData\Local\Python\pythoncore-3.14-64\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\Users\semee\AppData\Local\Python\pythoncore-3.14-64\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


<class 'numpy.ndarray'>
Input tensor shape: torch.Size([44, 1, 1, 8192])
Labels: (tensor([0, 1]), tensor([22, 22]))


In [275]:
class EEGNet(nn.Module):
    """
    Simplified EEGNet for P300 classification.
    
    Based on: Lawhern et al. (2018) "EEGNet: A Compact Convolutional
    Neural Network for EEG-based Brain-Computer Interfaces"
    """
    def __init__(self, n_channels=8, n_timepoints=250, n_classes=2,
                 F1=8, D=2, F2=16, dropout_rate=0.5):
        super().__init__()
        
        # Block 1: Temporal convolution + Depthwise spatial convolution
        self.block1 = nn.Sequential(
            # Temporal convolution (learns frequency filters)
            nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1),
            # Depthwise spatial convolution (learns spatial filters)
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout_rate),
        )
        
        # Block 2: Separable convolution
        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F2, (1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout_rate),
        )
        
        # Calculate the flattened size
        # After block1: (F1*D, 1, n_timepoints//4)
        # After block2: (F2, 1, n_timepoints//32)
        flat_size = F2 * (n_timepoints // 32)
        
        self.classifier = nn.Linear(flat_size, n_classes)
    
    def forward(self, x):
        # x shape: (batch, 1, n_channels, n_timepoints)
        x = self.block1(x)
        x = self.block2(x)
        x = x.flatten(1)
        return self.classifier(x)

In [288]:
class IDTest(nn.Module):
    def __init__(self, n_channels=1, n_timepoints=8192, n_classes=2):
        super().__init__()

        self.block1 = nn.Linear(8192, 2048)
        self.block2 = nn.Linear(2048, 1)

    def forward(self, x):
        x = self.block1(x)
        x = F.relu(x)
        x = self.block2(x)
        return x

In [289]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

print(f'PyTorch version: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch version: 2.11.0+cpu
Using device: cpu


In [290]:
n_channels = 1
n_timepoints = 8192
model = IDTest(n_channels=n_channels, n_timepoints=n_timepoints).to(device)
loss = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=.004)

N_EPOCHS = 100
train_losses = []
test_accs = []

for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0
    for x, y in train_dataset:
        optimizer.zero_grad()
        output = model(x)
        output = output.to(torch.float32)
        print(outputs)
        loss = loss(outputs, y)
        print(loss)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_loader))
    
    # Evaluate on test set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)

            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
    
    test_acc = correct / total
    test_accs.append(test_acc)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {train_losses[-1]:.4f} | Test Acc: {test_acc:.3f}')

print(f'\nFinal Test Accuracy: {test_accs[-1]:.3f}')

tensor([[[-4.2479]]], grad_fn=<ViewBackward0>)
tensor(27.5403, grad_fn=<MseLossBackward0>)


RuntimeError: Found dtype Long but expected Float

In [ ]:
class EEGID(nn.Module):
    def __init__(self):
        super(EEGID, self).__init__()

        self.fc1 = nn.Linear(672528, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output



In [ ]:
#Train & Implement
#Heavily inspired from Lab 4 Code

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

N_EPOCHS = 50
train_losses = []
test_accs = []

for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_loader))
    
    # Evaluate on test set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
    
    test_acc = correct / total
    test_accs.append(test_acc)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {train_losses[-1]:.4f} | Test Acc: {test_acc:.3f}')

print(f'\nFinal Test Accuracy: {test_accs[-1]:.3f}')